# MedNorm-VI E4 PhoBERT W2NER Training

Intended environment: Google Colab with Google Drive mounted. This notebook runs E4 smoke training by default and can run full training only after the explicit authorization string is set. Artifacts are written under `/content/drive/MyDrive/MedNorm-VI/artifacts/`; model caches stay outside artifact directories. It never runs organizer inference, never writes `output.zip`, and never accesses internal_test.

## Audit 0038 input contract

The W2NER relation grid is indexed by **atomic original-text words**, not by VnCoreNLP segmented model words. VnCoreNLP output is used only as PhoBERT input. A real governed entity proved the two coordinate systems must be decoupled:

```text
vimedner:train:train-000054   (validation split, row 4)
gold SYMPTOM 85:102 "rối loạn nhịp tim"
VnCoreNLP model word "gây_rối" spans 81:88, so the gold start 85 fell INSIDE one model word
```

Atomic words give `gây 81:84` and `rối 85:88`, so the entity aligns to `rối | loạn | nhịp | tim`. An Audit-0037 checkpoint describes a different input space and is rejected, not resumed.

## Run-all order (one fresh `Runtime -> Run all`)

1. mount Drive; 2. clone/update repository; 3. repo root + `PYTHONPATH`; 4. resolve governed corpus by authoritative SHA-256; 5. resolve immutable model/tokenizer revisions; 6. acquire **tokenizer only**; 7. atomic original-word surfaces; 8. VnCoreNLP model-word surfaces; 9. validate the atomic/model-word projection; 10. complete train + validation corpus preflight; 11. print and save the diagnostic summary; 12. **only if preflight passes**, acquire the 1.48 GB encoder; 13. smoke/full training; 14. save, reload, hash and validate the artifact.

The large encoder is never downloaded or instantiated before the full alignment preflight passes. VnCoreNLP is initialized once per process and is not reset when a cell is rerun.


In [ ]:
from pathlib import Path
import json
import os
import random
import re
import subprocess
import sys
import time

DRIVE_ROOT = Path("/content/drive/MyDrive/MedNorm-VI")
REPO_DIR = Path("/content/MedNorm-VI")
REPO_URL = os.environ.get("MEDNORM_REPO_URL", "https://github.com/vquclinh/MedNorm-VI")
REPO_REF = os.environ.get("MEDNORM_REPO_REF", "main")
CORPUS_DIR = DRIVE_ROOT / "data"
# Audit-0039 regression smoke writes to a FRESH directory so its evidence is not
# mixed with the validated Audit-0038 smoke artifact (…_smoke_v1), which is kept
# as historical runtime evidence and is never an initializer.
SMOKE_OUTPUT_DIR = DRIVE_ROOT / "artifacts" / "e4_phobert_w2ner_smoke_v2"
ARCHIVED_SMOKE_OUTPUT_DIR = DRIVE_ROOT / "artifacts" / "e4_phobert_w2ner_smoke_v1"
FULL_OUTPUT_DIR = DRIVE_ROOT / "artifacts" / "e4_phobert_w2ner_full_v1"
MODEL_CACHE_DIR = DRIVE_ROOT / "model_cache" / "huggingface"
VNCORENLP_DIR = DRIVE_ROOT / "model_cache" / "vncorenlp"

# LOCAL (non-Drive) runtime storage. The governed corpus is copied here once and
# every later read is local, so a Drive FUSE interruption cannot kill a long run.
# Checkpoints are also staged here and only then synced to Drive.
RUNTIME_ROOT = Path("/content/mednorm_vi_runtime")
RUNTIME_SPLITS_DIR = RUNTIME_ROOT / "splits"
STAGING_DIR = RUNTIME_ROOT / "staging"
RUNTIME_LOGS_DIR = RUNTIME_ROOT / "logs"
# ---------------------------------------------------------------------------
# OPERATOR SETTINGS
#
# The values below are the SMOKE defaults and are safe for a fresh Run all.
# The notebook is never committed with full training enabled.
#
#   SMOKE  : run smoke True, run full False, CONFIRM_FULL empty,
#            both resume flags False.
#   FULL   : run smoke False, run full True, CONFIRM_FULL set to the
#            authorization string E4_FULL_AUTHORIZATION, smoke-resume False
#            (never permitted for a full run), full-resume False for a fresh run.
#   RESUME : for an INTERRUPTED full run only, set the full-resume flag True.
#            It is accepted only against a compatible full checkpoint, which
#            must match the input contract, checkpoint schema, atomic
#            projection, config hash, model and tokenizer revisions, weight
#            format, precision policy, optimizer and accumulation settings.
# ---------------------------------------------------------------------------
RUN_SMOKE_TRAINING = True
RUN_FULL_TRAINING = False
CONFIRM_FULL = ""
RESUME_FROM_SMOKE_CHECKPOINT = False
RESUME_FROM_FULL_CHECKPOINT = False
SEED = 20260727
SMOKE_EPOCHS = 1
FULL_EPOCHS = 12

# Real gradient accumulation (Audit 0039). W2NER grids are variable-sized per
# document, so the micro-batch is one document and the effective batch is reached
# by accumulation. EFFECTIVE_BATCH_SIZE is DERIVED, never declared independently.
MICRO_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8
EFFECTIVE_BATCH_SIZE = MICRO_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.0
MAX_GRAD_NORM = 1.0
OPTIMIZER_NAME = "AdamW"
REQUESTED_PRECISION = "bf16"

MAX_WORDS = 256
MAX_MODEL_TOKENS = 512
SMOKE_ROWS = 8

# ---------------------------------------------------------------------------
# PROGRESS OBSERVABILITY (Audit 0041)
#
# A real T4 run completed its first micro-batch and then printed nothing for
# roughly an hour: there was simply no logging between the first sample and the
# end-of-epoch summary, so progress was indistinguishable from a hang.
#
# The first ten samples are logged individually so motion is visible within
# seconds. After that the interval is 100 — NOT 5. Five would emit ~81,000
# notebook lines across 405,912 backward passes and read GPU counters far more
# often than useful. An operator may temporarily set
# PROGRESS_LOG_EVERY_N_TRAIN_SAMPLES = 5 for a smoke/debug run; training
# semantics are unaffected either way.
# ---------------------------------------------------------------------------
PROGRESS_ENABLED = True
PROGRESS_LOG_FIRST_N_SAMPLES = 10
PROGRESS_LOG_EVERY_N_TRAIN_SAMPLES = 100
PROGRESS_LOG_EVERY_N_VALIDATION_SAMPLES = 50
PROGRESS_BAR_ENABLED = True
PROGRESS_ROLLING_LOSS_WINDOW = 100
BOOTSTRAP_DEPENDENCIES = (
    "transformers>=4.41",
    "huggingface_hub>=0.23",
    "safetensors>=0.4",
    "py_vncorenlp>=0.1.4",
)

try:
    from google.colab import drive, userdata
except ModuleNotFoundError:
    drive = None
    userdata = None

if drive is None:
    raise RuntimeError("This training notebook is intended for Colab; google.colab.drive is unavailable")
drive.mount("/content/drive")
if not DRIVE_ROOT.exists():
    raise RuntimeError(f"Drive root is unavailable after mount: {DRIVE_ROOT}")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "fetch", "origin", "--prune"], cwd=REPO_DIR, check=True)
subprocess.run(["git", "checkout", REPO_REF], cwd=REPO_DIR, check=True)
if not re.fullmatch(r"[0-9a-f]{40}", REPO_REF):
    subprocess.run(["git", "pull", "--ff-only"], cwd=REPO_DIR, check=True)
RESOLVED_COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True).strip()
if not re.fullmatch(r"[0-9a-f]{40}", RESOLVED_COMMIT):
    raise AssertionError("repository commit must be a 40-hex SHA")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", *BOOTSTRAP_DEPENDENCIES], check=True)
os.chdir(REPO_DIR)
repo_src = (REPO_DIR / "src").resolve()
if str(repo_src) not in sys.path:
    sys.path.insert(0, str(repo_src))
os.environ["PYTHONPATH"] = str(repo_src) + os.pathsep + os.environ.get("PYTHONPATH", "")

import mednorm_vi  # noqa: E402
module_path = Path(mednorm_vi.__file__).resolve()
if repo_src not in module_path.parents:
    raise RuntimeError(f"mednorm_vi imported from {module_path}, not from cloned repo {repo_src}")

random.seed(SEED)
LOCAL_PROTOCOL_ASSERTION = dict(internal_test_accessed=False)
print(json.dumps({
    "stage": "bootstrap",
    "repository_commit": RESOLVED_COMMIT,
    "repo_dir": str(REPO_DIR),
    "mednorm_vi_import": str(module_path),
}, indent=2, sort_keys=True))

In [ ]:
from mednorm_vi.mention_factory.w2ner import (
    ATOMIC_WORD_POLICY_VERSION,
    EntitySpan,
    build_relation_grid_head,
    decode_w2ner_grid,
    tokenize_atomic_words,
)
from mednorm_vi.training.phase2.artifacts import STATUS_FULLY_TRAINED, STATUS_SMOKE_EXECUTED, validate_e4_artifact
from mednorm_vi.training.phase2.common import canonical_json_sha256, sha256_file
from mednorm_vi.training.phase2.e4_alignment_diagnostic import run_alignment_diagnostic
from mednorm_vi.training.phase2.e4_progress import (
    DEFAULT_PROGRESS_LOG_NAME,
    PERSISTENCE_PHASES,
    ProgressConfig,
    ProgressLog,
    RollingLoss,
    emit,
    epoch_checkpoint_persisted_record,
    epoch_start_record,
    epoch_training_complete_record,
    epoch_validation_complete_record,
    eta_seconds,
    full_training_complete_record,
    gpu_memory_snapshot,
    make_progress_bar,
    persistence_phase_record,
    progress_bar_postfix,
    rate_per_second,
    should_log_train_sample,
    should_log_validation_sample,
    training_failed_record,
    training_heartbeat,
    validation_heartbeat,
)
from mednorm_vi.training.phase2.e4_runtime_io import (
    ArtifactPersistenceError,
    BoundedAlignmentReport,
    DriveAdapter,
    DriveHealthReport,
    GovernedW2NERContractSource,
    assert_not_materialized,
    ensure_drive_healthy,
    materialization_summary,
    materialize_governed_splits,
    memory_snapshot,
    persist_artifact,
)
from mednorm_vi.training.phase2.e4_w2ner_training import (
    ATOMIC_PROJECTION_VERSION,
    E4_INPUT_CONTRACT_VERSION,
    E4_FULL_AUTHORIZATION,
    assert_compatible_full_resume,
    assert_full_initialization_source,
    assert_full_training_device,
    assert_optimizer_step_accounting,
    assert_weight_format_loadable,
    build_e4_history_row,
    build_e4_training_accounting,
    build_e4_training_state_payload,
    optimizer_signature,
    plan_gradient_accumulation,
    resolve_mixed_precision_policy,
    resolve_phobert_weight_format,
    E4_GOVERNED_TRAIN_SHA256,
    E4_GOVERNED_VALIDATION_SHA256,
    E4_MODEL_ID,
    assert_full_not_initialized_from_smoke,
    build_e4_manifest,
    build_e4_resolved_config,
    atomic_relation_head_input_dim,
    build_atomic_projection,
    build_w2ner_batch_contract_from_segmented_words,
    decode_w2ner_logits,
    prepare_phobert_word_inputs,
    project_to_atomic_word_embeddings,
    reject_incompatible_e4_checkpoint,
    resolve_e4_governed_splits,
    validate_phobert_encoder_load_report,
)
from mednorm_vi.training.phobert_alignment import (
    map_segmented_words,
    resolve_segmented_text,
    segmented_text_to_words,
    verify_tokenizer_equivalence,
)

# ---------------------------------------------------------------------------
# STEP 2 (helper form): Drive health check.
#
# The basic mount happened in the bootstrap cell because the health helper lives
# in the repository, which is only cloned there. This is the first use of the
# tracked probe, and it runs BEFORE any governed corpus I/O.
#
# A real A100 run died at exactly this boundary with:
#     OSError: [Errno 107] Transport endpoint is not connected
# while reopening the governed train split on the long-lived Drive FUSE mount.
# ---------------------------------------------------------------------------
def _colab_drive_adapter() -> DriveAdapter:
    return DriveAdapter(
        mount=lambda: drive.mount("/content/drive", force_remount=True),
        flush_and_unmount=getattr(drive, "flush_and_unmount", None),
        force_unmount=None,
    )

def check_drive_health(probe_files=()) -> DriveHealthReport:
    """One bounded remount at most; a healthy Drive is never remounted."""
    return ensure_drive_healthy(
        "/content/drive", DRIVE_ROOT,
        adapter=_colab_drive_adapter(),
        probe_files=tuple(probe_files),
        max_remount_attempts=1,
    )

DRIVE_HEALTH = check_drive_health()
print(json.dumps({"stage": "drive_health_check", **DRIVE_HEALTH.as_dict()},
                 indent=2, sort_keys=True))


In [ ]:
# ---------------------------------------------------------------------------
# STEPS 5-7: resolve the governed corpus on Drive, validate its source hashes,
# then MATERIALIZE only train and validation to local /content storage and switch
# the active paths to the local copies.
#
# Everything after this point reads the corpus locally. The governed SHA-256
# values stay authoritative: a copy is accepted only on an exact digest match, so
# materialization cannot change examples, ordering, text, offsets, entities or
# split identity. internal_test is never copied and is refused by name.
# ---------------------------------------------------------------------------
EXPECTED_CORPUS_HASHES = {
    "train": E4_GOVERNED_TRAIN_SHA256,
    "validation": E4_GOVERNED_VALIDATION_SHA256,
}

def validate_corpus_hashes(corpus_dir: Path) -> dict[str, str]:
    global split_resolutions
    search_roots = (
        corpus_dir,
        corpus_dir / "processed",
        DRIVE_ROOT / "data" / "derived",
        REPO_DIR / "data",
    )
    split_resolutions = resolve_e4_governed_splits(search_roots)
    observed = {name: resolution.sha256 for name, resolution in split_resolutions.items()}
    if observed != EXPECTED_CORPUS_HASHES:
        raise AssertionError("governed corpus hashes do not match authoritative E4 splits")
    return observed

corpus_hashes = validate_corpus_hashes(CORPUS_DIR)
DRIVE_TRAIN_SPLIT_PATH = split_resolutions["train"].path
DRIVE_VALIDATION_SPLIT_PATH = split_resolutions["validation"].path

# Probe Drive against the real source files before the long read.
DRIVE_HEALTH = check_drive_health(
    probe_files=(DRIVE_TRAIN_SPLIT_PATH, DRIVE_VALIDATION_SPLIT_PATH))

GOVERNED_SOURCES = {
    "train": (DRIVE_TRAIN_SPLIT_PATH, E4_GOVERNED_TRAIN_SHA256),
    "validation": (DRIVE_VALIDATION_SPLIT_PATH, E4_GOVERNED_VALIDATION_SHA256),
}
MATERIALIZED_SPLITS = materialize_governed_splits(GOVERNED_SOURCES, RUNTIME_SPLITS_DIR)
MATERIALIZATION = materialization_summary(MATERIALIZED_SPLITS, GOVERNED_SOURCES)

# STEP 7: the active paths are now LOCAL. No later stage reopens Drive for corpus.
TRAIN_SPLIT_PATH = Path(MATERIALIZED_SPLITS["train"].runtime_path)
VALIDATION_SPLIT_PATH = Path(MATERIALIZED_SPLITS["validation"].runtime_path)
for _path in (TRAIN_SPLIT_PATH, VALIDATION_SPLIT_PATH):
    if str(_path).startswith("/content/drive"):
        raise AssertionError(f"active corpus path is still on Drive: {_path}")

print(json.dumps({
    "stage": "governed_corpus_local_materialization",
    "corpus_hashes": corpus_hashes,
    "materialization": MATERIALIZATION,
    "drive_health": DRIVE_HEALTH.as_dict(),
    "memory": memory_snapshot("after_materialization"),
    "internal_test_accessed": False,
}, indent=2, sort_keys=True))


In [ ]:
IMMUTABLE_REVISION_RE = re.compile(r"[0-9a-f]{40}")

def optional_hf_token() -> str | None:
    token = os.environ.get("HF_TOKEN", "").strip()
    if token:
        return token
    if userdata is None:
        return None
    try:
        secret = userdata.get("HF_TOKEN")
    except Exception:
        return None
    return str(secret).strip() or None

def resolve_hf_revision(model_id: str, *, env_var: str, requested_revision: str = "main") -> str:
    env_revision = os.environ.get(env_var, "").strip()
    if env_revision:
        if not IMMUTABLE_REVISION_RE.fullmatch(env_revision):
            raise SystemExit(f"{env_var} must be an immutable 40-hex revision, not {env_revision!r}")
        return env_revision
    from huggingface_hub import HfApi
    info = HfApi(token=optional_hf_token()).model_info(model_id, revision=requested_revision)
    resolved = str(info.sha)
    if not IMMUTABLE_REVISION_RE.fullmatch(resolved):
        raise SystemExit(f"Hugging Face did not resolve {model_id} to an immutable 40-hex revision")
    return resolved

PINNED_MODEL_REVISION = resolve_hf_revision(E4_MODEL_ID, env_var="MEDNORM_E4_MODEL_REVISION")
PINNED_TOKENIZER_REVISION = os.environ.get("MEDNORM_E4_TOKENIZER_REVISION", "").strip() or PINNED_MODEL_REVISION

def require_resolved_revision(value: str, field_name: str) -> None:
    if not IMMUTABLE_REVISION_RE.fullmatch(value):
        raise SystemExit(f"{field_name} must be resolved before the revision gate")

require_resolved_revision(PINNED_MODEL_REVISION, "PINNED_MODEL_REVISION")
require_resolved_revision(PINNED_TOKENIZER_REVISION, "PINNED_TOKENIZER_REVISION")
print(json.dumps({
    "stage": "revision_resolution",
    "model_id": E4_MODEL_ID,
    "model_revision": PINNED_MODEL_REVISION,
    "tokenizer_revision": PINNED_TOKENIZER_REVISION,
    "hf_token_present": optional_hf_token() is not None,
}, indent=2, sort_keys=True))

In [ ]:
assert_full_not_initialized_from_smoke(
    run_full_training=RUN_FULL_TRAINING,
    resume_from_smoke_checkpoint=RESUME_FROM_SMOKE_CHECKPOINT,
)
INITIALIZATION_SOURCE = assert_full_initialization_source(
    run_full_training=RUN_FULL_TRAINING,
    resume_from_smoke_checkpoint=RESUME_FROM_SMOKE_CHECKPOINT,
    resume_from_full_checkpoint=RESUME_FROM_FULL_CHECKPOINT,
)
if RUN_FULL_TRAINING and CONFIRM_FULL != E4_FULL_AUTHORIZATION:
    raise SystemExit("E4 full training requires explicit operator authorization")
if not (RUN_SMOKE_TRAINING or RUN_FULL_TRAINING):
    raise SystemExit("Run all defaults to smoke; set RUN_SMOKE_TRAINING=True or RUN_FULL_TRAINING=True")

OUTPUT_DIR = FULL_OUTPUT_DIR if RUN_FULL_TRAINING else SMOKE_OUTPUT_DIR
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "checkpoints").mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "logs").mkdir(parents=True, exist_ok=True)
MODEL_CACHE_DIR.mkdir(parents=True, exist_ok=True)
VNCORENLP_DIR.mkdir(parents=True, exist_ok=True)
# Local staging: every artifact is written and verified here before it is synced
# to Drive, so a transport failure cannot destroy an epoch's work.
STAGING_DIR.mkdir(parents=True, exist_ok=True)
(STAGING_DIR / "checkpoints").mkdir(parents=True, exist_ok=True)
(STAGING_DIR / "logs").mkdir(parents=True, exist_ok=True)
RUNTIME_LOGS_DIR.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(MODEL_CACHE_DIR)
mode = "full" if RUN_FULL_TRAINING else "smoke"
EPOCHS = FULL_EPOCHS if RUN_FULL_TRAINING else SMOKE_EPOCHS

# The pinned official PhoBERT revision publishes pytorch_model.bin only; a real
# Colab run failed on use_safetensors=True. Resolved once, before acquisition.
WEIGHT_FORMAT = resolve_phobert_weight_format(E4_MODEL_ID, PINNED_MODEL_REVISION)
assert_weight_format_loadable(WEIGHT_FORMAT)

OPTIMIZER_SIGNATURE = optimizer_signature(
    name=OPTIMIZER_NAME, learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY, max_grad_norm=MAX_GRAD_NORM,
)
# resolved_config is finalized AFTER the corpus preflight, because the
# accumulation plan needs the real training example count and the precision
# policy needs the real device. See the "resolved execution summary" cell.
print(json.dumps({
    "stage": "run_gate",
    "mode": mode,
    "output_dir": str(OUTPUT_DIR),
    "full_authorized": RUN_FULL_TRAINING and CONFIRM_FULL == E4_FULL_AUTHORIZATION,
    "initialization_source": INITIALIZATION_SOURCE,
    "pretrained_weight_format": WEIGHT_FORMAT.filename,
    "use_safetensors": WEIGHT_FORMAT.use_safetensors,
    "internal_test_accessed": False,
}, indent=2, sort_keys=True))

In [ ]:
# ---------------------------------------------------------------------------
# STEP 6: acquire the TOKENIZER ONLY. The 1.48 GB encoder is not touched here.
# ---------------------------------------------------------------------------
from transformers import AutoTokenizer

USE_FAST_TOKENIZER = os.environ.get("MEDNORM_E4_USE_FAST_TOKENIZER", "0") == "1"
tokenizer = AutoTokenizer.from_pretrained(
    E4_MODEL_ID,
    revision=PINNED_TOKENIZER_REVISION,
    cache_dir=str(MODEL_CACHE_DIR),
    use_fast=USE_FAST_TOKENIZER,
    token=optional_hf_token(),
    local_files_only=False,
)
tokenizer_report = {
    "tokenizer_class": type(tokenizer).__name__,
    "tokenizer_is_fast": bool(getattr(tokenizer, "is_fast", False)),
    "slow_path_required_for_official_phobert": not bool(getattr(tokenizer, "is_fast", False)),
}

# VnCoreNLP SINGLETON. py_vncorenlp starts a JVM and chdirs into save_dir; starting
# a second one in the same process raises. The annotator is cached in globals() so a
# cell rerun reuses it instead of re-initializing, and the working directory is
# restored immediately.
if globals().get("VNCORENLP_ANNOTATOR") is None:
    import py_vncorenlp
    VNCORENLP_DIR.mkdir(parents=True, exist_ok=True)
    if not any(VNCORENLP_DIR.glob("*.jar")):
        py_vncorenlp.download_model(save_dir=str(VNCORENLP_DIR))
    _cwd_before_jvm = Path.cwd()
    try:
        VNCORENLP_ANNOTATOR = py_vncorenlp.VnCoreNLP(annotators=["wseg"], save_dir=str(VNCORENLP_DIR))
    finally:
        os.chdir(_cwd_before_jvm)
    VNCORENLP_INITIALIZED_ONCE = True
else:
    VNCORENLP_INITIALIZED_ONCE = False

def segment_with_vncorenlp(text: str) -> str:
    segments = VNCORENLP_ANNOTATOR.word_segment(text)
    if not segments:
        raise RuntimeError("VnCoreNLP returned no segments")
    return " ".join(segments)

print(json.dumps({
    "stage": "tokenizer_and_segmenter_acquisition",
    **tokenizer_report,
    "vncorenlp_initialized_this_run": VNCORENLP_INITIALIZED_ONCE,
    "encoder_downloaded": False,
}, indent=2, sort_keys=True))


In [ ]:
# ---------------------------------------------------------------------------
# STEPS 7-9: atomic original-word surface, VnCoreNLP model-word surface, and the
# projection between them. Still no encoder.
# ---------------------------------------------------------------------------
def entity_from_row(ent: dict) -> EntitySpan:
    entity_type = ent.get("target_type") or ent.get("type") or ent.get("label")
    if entity_type is None:
        raise RuntimeError("governed entity has no organizer type field")
    return EntitySpan(int(ent["start"]), int(ent["end"]), str(entity_type), str(ent["text"]))

def build_surfaces(text: str):
    """(atomic grid words, VnCoreNLP model words) for one governed example."""
    atomic_words = tokenize_atomic_words(text)                       # STEP 7
    segmented_text, segmentation_source = resolve_segmented_text(text, segment_with_vncorenlp)
    model_words = map_segmented_words(text, segmented_text_to_words(segmented_text))  # STEP 8
    return atomic_words, model_words, segmentation_source

# STEP 9: the projection must hold on the exact example that failed in Colab.
PROBE_TEXT = (
    "tìm kiếm các dấu hiệu của các bệnh khác , chẳng hạn như bệnh tuyến giáp , "
    "có thể gây rối loạn nhịp tim ."
)
probe_atomic, probe_model, _probe_source = build_surfaces(PROBE_TEXT)
probe_entity = EntitySpan(85, 102, "SYMPTOM", PROBE_TEXT[85:102])
if probe_entity.text != "rối loạn nhịp tim":
    raise AssertionError("probe entity text drifted from the audited Colab failure")
probe_contract = build_w2ner_batch_contract_from_segmented_words(
    "probe", PROBE_TEXT, (probe_entity,), probe_model, max_words=MAX_WORDS,
)
probe_encoding = prepare_phobert_word_inputs(tokenizer, probe_contract.segmented_words, max_length=MAX_MODEL_TOKENS)
probe_projection = build_atomic_projection(
    PROBE_TEXT, probe_contract.segmented_words, probe_encoding, atomic_words=probe_contract.atomic_words,
)
probe_decoded = {(s.start, s.end, s.entity_type) for s in decode_w2ner_grid(probe_contract.grid)}
if (85, 102, "SYMPTOM") not in probe_decoded:
    raise AssertionError("atomic grid failed to represent the audited governed entity")
probe_features = {
    word.text: probe_projection.atomic_features[index]
    for index, word in enumerate(probe_projection.atomic_words)
    if word.text in {"gây", "rối"}
}
if len(probe_features) != 2 or len(set(probe_features.values())) != 2:
    raise AssertionError("atomic words under one merged model token must stay distinguishable")
print(json.dumps({
    "stage": "surface_and_projection_validation",
    "atomic_word_count": len(probe_atomic),
    "model_word_count": len(probe_model),
    "audited_entity_representable": True,
    "merged_model_words": probe_projection.merged_model_word_count,
    "multi_model_atomic_words": probe_projection.multi_model_word_atomic_count,
    "gay_roi_features": {key: list(value) for key, value in sorted(probe_features.items())},
    "input_contract_version": E4_INPUT_CONTRACT_VERSION,
    "atomic_projection_version": ATOMIC_PROJECTION_VERSION,
    "encoder_downloaded": False,
}, indent=2, sort_keys=True))


In [ ]:
# ---------------------------------------------------------------------------
# STEPS 9-10: complete alignment preflight FROM LOCAL FILES, then bounded
# metadata only. No full-corpus contract list is ever built.
# ---------------------------------------------------------------------------
print(json.dumps({"stage": "memory_before_preflight",
                  "memory": memory_snapshot("before_preflight")},
                 indent=2, sort_keys=True))

alignment_diagnostic = run_alignment_diagnostic(
    {"train": TRAIN_SPLIT_PATH, "validation": VALIDATION_SPLIT_PATH},
    segmenter=segment_with_vncorenlp,
    tokenizer=tokenizer,
    max_words=MAX_WORDS,
    max_model_tokens=MAX_MODEL_TOKENS,
)
DIAGNOSTIC_STAGING = STAGING_DIR / "e4_alignment_diagnostic.json"
DIAGNOSTIC_SHA256 = alignment_diagnostic.write(DIAGNOSTIC_STAGING)
diagnostic_summary = alignment_diagnostic.summary()
print(json.dumps({
    "stage": "full_corpus_alignment_preflight",
    "diagnostic_sha256": DIAGNOSTIC_SHA256,
    "summary": diagnostic_summary,
    "memory": memory_snapshot("after_preflight"),
    "internal_test_accessed": False,
    "encoder_downloaded": False,
}, indent=2, sort_keys=True))

# Governed policy: every entity must align to atomic word boundaries, with no
# snapping, no trimming and no silent exclusion.
if not alignment_diagnostic.passed:
    raise AssertionError(
        "E4 alignment preflight failed: "
        f"{alignment_diagnostic.unalignable_after_atomic} unalignable entities, "
        f"{alignment_diagnostic.silent_exclusions} exclusions, "
        f"{alignment_diagnostic.projection_violations} projection violations"
    )
PREFLIGHT_PASSED = True

# --- one governed contract, built on demand -------------------------------
def build_governed_contract(split: str, row: dict):
    """Build ONE W2NER contract plus a small report. Nothing is retained here."""
    text = str(row["text"])
    entities = tuple(entity_from_row(ent) for ent in row.get("entities", []))
    _atomic, model_words, segmentation_source = build_surfaces(text)
    verify_tokenizer_equivalence(model_words, tokenizer)
    contract = build_w2ner_batch_contract_from_segmented_words(
        str(row.get("example_id", row.get("id", ""))),
        text, entities, model_words, max_words=MAX_WORDS,
    )
    encoding = prepare_phobert_word_inputs(
        tokenizer, contract.segmented_words, max_length=MAX_MODEL_TOKENS)
    if "offset_mapping" in encoding.model_inputs:
        raise RuntimeError("offset_mapping leaked into encoder inputs")
    projection = build_atomic_projection(
        text, contract.segmented_words, encoding, atomic_words=contract.atomic_words)
    if projection.atomic_word_count != contract.word_count:
        raise RuntimeError("projection and W2NER grid disagree on the atomic word count")
    report = {
        "split": split,
        "segmentation_source": segmentation_source,
        "atomic_word_count": contract.word_count,
        "model_word_count": len(contract.segmented_words),
        "encoded_tokens": len(encoding.model_inputs["input_ids"]),
        "label_count": contract.label_count,
        "tokenizer_is_fast": encoding.tokenizer_is_fast,
        "consumed_offset_mapping": encoding.consumed_offset_mapping,
    }
    return contract, report

def governed_source(split: str, path: Path, expected_sha256: str, example_count: int):
    """A REPEATABLE streaming source; a fresh iterator per epoch / validation pass."""
    return GovernedW2NERContractSource(
        split=split,
        path=str(path),
        expected_sha256=expected_sha256,
        expected_example_count=example_count,
        tokenizer=tokenizer,
        build_contract=build_governed_contract,
        max_words=MAX_WORDS,
        max_model_tokens=MAX_MODEL_TOKENS,
        input_contract_version=E4_INPUT_CONTRACT_VERSION,
        max_rows=None if RUN_FULL_TRAINING else SMOKE_ROWS,
    )

_diagnostic_splits = alignment_diagnostic.as_dict()["by_split"]
TRAIN_SOURCE = governed_source(
    "train", TRAIN_SPLIT_PATH, E4_GOVERNED_TRAIN_SHA256,
    int(_diagnostic_splits["train"]["examples"]))
VALIDATION_SOURCE = governed_source(
    "validation", VALIDATION_SPLIT_PATH, E4_GOVERNED_VALIDATION_SHA256,
    int(_diagnostic_splits["validation"]["examples"]))
TRAIN_SOURCE.verify_identity()
VALIDATION_SOURCE.verify_identity()

# Structural guard: full mode must never hand training a materialized list.
assert_not_materialized(TRAIN_SOURCE, label="TRAIN_SOURCE")
assert_not_materialized(VALIDATION_SOURCE, label="VALIDATION_SOURCE")

# Bounded statistics: one pass, a three-item sample, scalars only.
train_report = BoundedAlignmentReport(sample_limit=3)
for _contract in TRAIN_SOURCE.iter_contracts(reporter=train_report):
    del _contract
validation_report = BoundedAlignmentReport(sample_limit=3)
for _contract in VALIDATION_SOURCE.iter_contracts(reporter=validation_report):
    del _contract

grid_target_statistics = {
    "train_contracts": train_report.examples,
    "validation_contracts": validation_report.examples,
    "max_atomic_words": max(train_report.max_atomic_words,
                            validation_report.max_atomic_words),
    "label_count": max(train_report.label_count, validation_report.label_count),
    "train_hash": corpus_hashes["train"],
    "validation_hash": corpus_hashes["validation"],
    "tokenizer_class": type(tokenizer).__name__,
    "tokenizer_is_fast": bool(getattr(tokenizer, "is_fast", False)),
    "input_contract_version": E4_INPUT_CONTRACT_VERSION,
    "grid_word_surface": ATOMIC_WORD_POLICY_VERSION,
    "diagnostic_sha256": DIAGNOSTIC_SHA256,
    "contract_stream_version": TRAIN_SOURCE.stream_version,
    "contracts_materialized": False,
}
STATISTICS_STAGING = STAGING_DIR / "grid_target_statistics.json"
STATISTICS_STAGING.write_text(
    json.dumps(grid_target_statistics, indent=2, sort_keys=True) + "\n", encoding="utf-8")
print(json.dumps({
    "stage": "bounded_contract_statistics",
    "grid_target_statistics": grid_target_statistics,
    "train_sample_reports": train_report.as_dict(),
    "validation_sample_reports": validation_report.as_dict(),
    "memory": memory_snapshot("after_bounded_statistics"),
    "internal_test_accessed": False,
}, indent=2, sort_keys=True))


In [ ]:
# ---------------------------------------------------------------------------
# RESOLVED EXECUTION SUMMARY - printed BEFORE the encoder is acquired.
# Finalizes resolved_config.json now that the real example count and the real
# device are known, so CONFIG_SHA256 covers accumulation and precision too.
# ---------------------------------------------------------------------------
import torch

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE_TYPE = DEVICE.type

# GPU RUNTIME POLICY: full training requires CUDA, but no particular GPU model.
# T4 High-RAM, L4 and A100 are all acceptable. The device NAME is reported for
# observability only and is never used to accept or reject a runtime.
GPU_NAME = ""
if DEVICE_TYPE == "cuda":
    try:
        GPU_NAME = torch.cuda.get_device_name(0)
    except Exception:  # noqa: BLE001 - purely informational
        GPU_NAME = "unknown"
if RUN_FULL_TRAINING:
    assert_full_training_device(DEVICE_TYPE)

# bf16 support is resolved from the runtime capability, never from the name:
# T4 (compute capability 7.5) has no bf16 and correctly falls back to fp16 with a
# GradScaler; L4 and A100 resolve to bf16.
BF16_SUPPORTED = bool(
    DEVICE_TYPE == "cuda"
    and getattr(torch.cuda, "is_bf16_supported", lambda: False)()
)
PRECISION = resolve_mixed_precision_policy(
    REQUESTED_PRECISION, device_type=DEVICE_TYPE, bf16_supported=BF16_SUPPORTED,
)
PROGRESS = ProgressConfig(
    enabled=PROGRESS_ENABLED,
    log_first_n_samples=PROGRESS_LOG_FIRST_N_SAMPLES,
    log_every_n_train_samples=PROGRESS_LOG_EVERY_N_TRAIN_SAMPLES,
    log_every_n_validation_samples=PROGRESS_LOG_EVERY_N_VALIDATION_SAMPLES,
    progress_bar_enabled=PROGRESS_BAR_ENABLED,
    rolling_loss_window=PROGRESS_ROLLING_LOSS_WINDOW,
)
PROGRESS_LOG_PATH = RUNTIME_LOGS_DIR / DEFAULT_PROGRESS_LOG_NAME
PROGRESS_LOG = ProgressLog(PROGRESS_LOG_PATH, enabled=PROGRESS.enabled)

ACCUMULATION = plan_gradient_accumulation(
    TRAIN_SOURCE.example_count,
    micro_batch_size=MICRO_BATCH_SIZE,
    accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    epochs=EPOCHS,
)

resolved_config = build_e4_resolved_config(
    mode=mode,
    model_revision=PINNED_MODEL_REVISION,
    tokenizer_revision=PINNED_TOKENIZER_REVISION,
    seed=SEED,
    max_words=MAX_WORDS,
    effective_batch_size=ACCUMULATION.effective_batch_size,
    weight_format=WEIGHT_FORMAT,
    accumulation=ACCUMULATION,
    precision=PRECISION,
    progress=PROGRESS.as_dict(),
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    max_grad_norm=MAX_GRAD_NORM,
    optimizer_name=OPTIMIZER_NAME,
)
CONFIG_STAGING = STAGING_DIR / "resolved_config.json"
CONFIG_STAGING.write_text(
    json.dumps(resolved_config, indent=2, sort_keys=True) + "\n", encoding="utf-8")
CONFIG_SHA256 = canonical_json_sha256(resolved_config)

RESUME_SOURCE = (
    "compatible_full_training_checkpoint" if RESUME_FROM_FULL_CHECKPOINT
    else "pinned_pretrained_base"
)
EXECUTION_SUMMARY = {
    "stage": "resolved_execution_summary",
    "run_mode": mode,
    "output_dir": str(OUTPUT_DIR),
    "model_revision": PINNED_MODEL_REVISION,
    "tokenizer_revision": PINNED_TOKENIZER_REVISION,
    "pretrained_weight_format": WEIGHT_FORMAT.filename,
    "use_safetensors": WEIGHT_FORMAT.use_safetensors,
    "micro_batch_size": ACCUMULATION.micro_batch_size,
    "gradient_accumulation_steps": ACCUMULATION.accumulation_steps,
    "effective_batch_size": ACCUMULATION.effective_batch_size,
    "epochs": ACCUMULATION.epochs,
    "training_examples": ACCUMULATION.example_count,
    "expected_optimizer_steps": ACCUMULATION.expected_optimizer_steps,
    "optimizer_steps_per_epoch": ACCUMULATION.optimizer_steps_per_epoch,
    "expected_backward_passes": ACCUMULATION.expected_backward_passes,
    "final_partial_group_size": ACCUMULATION.final_partial_group_size,
    "gradient_clipping_max_norm": MAX_GRAD_NORM,
    "mixed_precision_policy": PRECISION.as_dict(),
    "device_type": DEVICE_TYPE,
    "progress": PROGRESS.as_dict(),
    "progress_log_path": str(PROGRESS_LOG_PATH),
    "gpu_name": GPU_NAME,
    "gpu_name_used_as_gate": False,
    "supported_runtimes": ["T4 High-RAM", "L4", "A100"],
    "resume_source": RESUME_SOURCE,
    "initialization_source": INITIALIZATION_SOURCE,
    "config_sha256": CONFIG_SHA256,
    "train_source": TRAIN_SOURCE.as_dict(),
    "validation_source": VALIDATION_SOURCE.as_dict(),
    "contracts_materialized": False,
    "corpus_read_from": "local_runtime",
    "drive_health": DRIVE_HEALTH.as_dict(),
    "materialization": MATERIALIZATION,
    "memory": memory_snapshot("before_encoder_acquisition"),
    "encoder_downloaded": False,
    "internal_test_accessed": False,
}
print(json.dumps(EXECUTION_SUMMARY, indent=2, sort_keys=True))


In [ ]:
def _gold_exact_set(item):
    return {(span.start, span.end, span.entity_type) for span in decode_w2ner_grid(item.grid)}

def _atomic_word_embeddings(base_model, tokenizer, item, device):
    """Encode one example and project PhoBERT states onto ATOMIC grid words."""
    import torch

    encoding = prepare_phobert_word_inputs(tokenizer, item.segmented_words, max_length=MAX_MODEL_TOKENS)
    if "offset_mapping" in encoding.model_inputs:
        raise RuntimeError("offset_mapping leaked into encoder inputs")
    projection = build_atomic_projection(
        item.grid.original_text, item.segmented_words, encoding, atomic_words=item.atomic_words,
    )
    if projection.atomic_word_count != item.word_count:
        raise RuntimeError("projection and W2NER grid disagree on the atomic word count")
    model_inputs = {
        key: torch.tensor([value], dtype=torch.long, device=device)
        for key, value in encoding.model_inputs.items()
    }
    outputs = base_model(**model_inputs)
    return project_to_atomic_word_embeddings(outputs.last_hidden_state[0], projection)

def _predict_exact_set(base_model, head, tokenizer, item, device):
    import torch
    word_embeddings = _atomic_word_embeddings(base_model, tokenizer, item, device)
    pair_mask = torch.tensor([item.grid.pair_mask], dtype=torch.bool, device=device)
    logits = head(word_embeddings, pair_mask)[0].detach().cpu().tolist()
    return set(decode_w2ner_logits(item, logits))

def evaluate_w2ner_validation(base_model, head, tokenizer, validation_source, device,
                              *, epoch: int = 0, total_epochs: int = 0) -> dict[str, float | int | bool]:
    """Stream a FRESH validation iterator; predictions are never accumulated.

    Validation over 1,045 examples can also look like a hang, so it reports a bar
    plus heartbeats at sample 1, every configured interval, and the last sample.
    Only COUNTS are reported - never predictions, gold spans or corpus text.
    """
    import torch
    base_model.eval()
    head.eval()
    true_positive = 0
    predicted_total = 0
    gold_total = 0
    total_samples = validation_source.example_count
    started = time.monotonic()
    bar = make_progress_bar(total_samples, f"validation epoch {epoch}/{total_epochs}",
                            enabled=PROGRESS.progress_bar_enabled)
    try:
        with torch.no_grad():
            # A new independent iterator for every validation pass.
            for index, item in enumerate(validation_source.iter_contracts()):
                sample = index + 1
                predicted = _predict_exact_set(base_model, head, tokenizer, item, device)
                gold = _gold_exact_set(item)
                true_positive += len(predicted & gold)
                predicted_total += len(predicted)
                gold_total += len(gold)
                del item, predicted, gold
                bar.update(1)
                if should_log_validation_sample(PROGRESS, sample, total_samples):
                    elapsed = time.monotonic() - started
                    bar.set_postfix(**progress_bar_postfix(
                        rolling_mean_loss=0.0, optimizer_steps=0,
                        samples_per_second=rate_per_second(sample, elapsed),
                        eta_seconds_value=eta_seconds(sample, total_samples, elapsed)))
                    emit(validation_heartbeat(
                        epoch=epoch, total_epochs=total_epochs, sample=sample,
                        total_samples=total_samples, elapsed_seconds=elapsed,
                        predicted_mentions_so_far=predicted_total,
                        gold_mentions_so_far=gold_total,
                        gpu_memory=gpu_memory_snapshot(torch, device.type),
                        memory_snapshot=memory_snapshot), log=PROGRESS_LOG)
    finally:
        bar.close()
    precision = true_positive / predicted_total if predicted_total else 0.0
    recall = true_positive / gold_total if gold_total else 0.0
    exact_f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {
        "validation_exact_precision": precision,
        "validation_exact_recall": recall,
        "validation_exact_f1": exact_f1,
        "validation_true_positive": true_positive,
        "validation_predicted_total": predicted_total,
        "validation_gold_total": gold_total,
        "internal_test_accessed": False,
    }

def checkpoint_state_payload(*, base_model, head, optimizer, scaler, epoch: int,
                             optimizer_steps: int, backward_passes: int,
                             examples_processed: int, best_metric: float,
                             best_checkpoint_sha256: str, parameter_count: int, mode: str):
    """Everything an exact full resume needs: weights, optimizer, scaler, counters."""
    return build_e4_training_state_payload(
        mode=mode,
        config_sha256=CONFIG_SHA256,
        model_revision=PINNED_MODEL_REVISION,
        tokenizer_revision=PINNED_TOKENIZER_REVISION,
        parameter_count=parameter_count,
        weight_format=WEIGHT_FORMAT,
        accumulation=ACCUMULATION,
        precision=PRECISION,
        optimizer_signature_value=OPTIMIZER_SIGNATURE,
        epoch=epoch,
        optimizer_steps=optimizer_steps,
        backward_passes=backward_passes,
        examples_processed=examples_processed,
        best_metric=best_metric,
        best_checkpoint_sha256=best_checkpoint_sha256,
        model_state={
            "base_model": base_model.state_dict(),
            "w2ner_head": head.state_dict(),
        },
        optimizer_state=optimizer.state_dict(),
        scaler_state=(scaler.state_dict() if scaler is not None else {}),
        scheduler_state=None,
    )

def validate_checkpoint_after_save_reload(path: Path, expected_sha256: str) -> None:
    import torch
    if not path.is_file():
        raise AssertionError(f"missing checkpoint: {path}")
    if sha256_file(path) != expected_sha256:
        raise AssertionError(f"checkpoint hash changed after save: {path}")
    payload = torch.load(path, map_location="cpu")
    required = {"checkpoint_schema_version", "expert_id", "mode", "config_sha256", "model_revision", "model_state"}
    missing = required - set(payload)
    if missing:
        raise AssertionError(f"checkpoint payload missing keys: {sorted(missing)}")
    if payload["config_sha256"] != CONFIG_SHA256:
        raise AssertionError("checkpoint config hash does not match resolved_config.json")
    # An Audit-0037 checkpoint describes the segmented-model-word grid and must
    # never be accepted or resumed under the atomic-grid contract.
    reject_incompatible_e4_checkpoint(payload)

def save_and_persist_checkpoint(name: str, payload, *, epoch: int = 0) -> str:
    """Save locally, prove it reloads, then sync to Drive with a verified digest.

    Order (Audit 0040): local torch.save -> local SHA-256 -> local reload check ->
    Drive health -> temporary persistent copy -> atomic replace -> persistent
    SHA-256 must equal the local one. A transport failure buys ONE bounded
    remount; after that the run stops with the verified local copy preserved
    instead of silently continuing to the next epoch.
    """
    import torch

    staged = STAGING_DIR / "checkpoints" / f"{name}.pt"
    # Named phases so a slow serialization or Drive sync never looks like a hang.
    emit(persistence_phase_record(PERSISTENCE_PHASES[0], epoch=epoch), log=PROGRESS_LOG)
    torch.save(payload, staged)
    digest = sha256_file(staged)
    emit(persistence_phase_record(PERSISTENCE_PHASES[1], epoch=epoch), log=PROGRESS_LOG)
    validate_checkpoint_after_save_reload(staged, digest)
    emit(persistence_phase_record(PERSISTENCE_PHASES[2], epoch=epoch), log=PROGRESS_LOG)
    try:
        emit(persistence_phase_record(PERSISTENCE_PHASES[3], epoch=epoch), log=PROGRESS_LOG)
        persisted = persist_artifact(
            f"{name}.pt", staged, OUTPUT_DIR / "checkpoints" / f"{name}.pt",
            drive_health=check_drive_health,
        )
        emit(persistence_phase_record(PERSISTENCE_PHASES[4], epoch=epoch), log=PROGRESS_LOG)
    except ArtifactPersistenceError as error:
        print(json.dumps({
            "stage": "persistent_checkpoint_custody_failed",
            "checkpoint": name,
            "local_staged_path": str(staged),
            "local_sha256": digest,
            "resumable_from_drive": False,
            "detail": str(error),
        }, indent=2, sort_keys=True))
        raise
    if persisted.sha256 != digest:
        raise AssertionError(f"{name}: persistent digest does not match the staged digest")
    emit(persistence_phase_record(PERSISTENCE_PHASES[5], epoch=epoch), log=PROGRESS_LOG)
    return digest


def run_training(train_source, validation_source, *, mode: str, epochs: int):
    import torch
    from torch import nn
    from transformers import AutoModel

    # STEP 12: the 1.48 GB encoder is acquired ONLY after the full-corpus alignment
    # preflight passed. Downloading it earlier wastes a Colab session on a run that
    # cannot build its training targets.
    if not globals().get("PREFLIGHT_PASSED", False):
        raise RuntimeError("refusing to acquire the PhoBERT encoder before the alignment preflight passes")
    # Full mode must never receive a materialized contract list.
    assert_not_materialized(train_source, label="train_source")
    assert_not_materialized(validation_source, label="validation_source")
    device = DEVICE
    if RUN_FULL_TRAINING:
        assert_full_training_device(device.type)
    # The pinned official revision publishes pytorch_model.bin only, so
    # use_safetensors is False by resolution. No manual edit is required.
    assert_weight_format_loadable(WEIGHT_FORMAT)
    base_model, loading_info = AutoModel.from_pretrained(
        E4_MODEL_ID,
        revision=PINNED_MODEL_REVISION,
        cache_dir=str(MODEL_CACHE_DIR),
        token=optional_hf_token(),
        local_files_only=False,
        use_safetensors=WEIGHT_FORMAT.use_safetensors,
        output_loading_info=True,
    )
    load_report = validate_phobert_encoder_load_report(
        missing_keys=tuple(loading_info.get("missing_keys", ())),
        unexpected_keys=tuple(loading_info.get("unexpected_keys", ())),
    )
    base_model.to(device)
    head = build_relation_grid_head(
        atomic_relation_head_input_dim(base_model.config.hidden_size),
        int(grid_target_statistics["label_count"]),
    ).to(device)
    parameter_count = sum(parameter.numel() for parameter in base_model.parameters()) + sum(parameter.numel() for parameter in head.parameters())
    trainable = list(base_model.parameters()) + list(head.parameters())
    optimizer = torch.optim.AdamW(
        trainable, lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scaler = torch.amp.GradScaler(device.type) if PRECISION.use_grad_scaler else None

    history_path = STAGING_DIR / "logs" / "training_history.jsonl"
    best_metric = -1.0
    optimizer_steps_total = 0
    backward_passes_total = 0
    examples_total = 0
    start_epoch = 1
    best_payload = None
    latest_payload = None
    best_checkpoint_sha256 = ""

    if RESUME_FROM_FULL_CHECKPOINT:
        # A resume must match the whole tracked training contract, precision and
        # accumulation included; otherwise the step accounting is meaningless.
        resume_path = STAGING_DIR / "checkpoints" / "latest.pt"
        if not resume_path.is_file():
            resume_path = OUTPUT_DIR / "checkpoints" / "latest.pt"
        resume_payload = torch.load(resume_path, map_location="cpu", weights_only=False)
        assert_compatible_full_resume(resume_payload, expected={
            "e4_input_contract_version": E4_INPUT_CONTRACT_VERSION,
            "e4_checkpoint_schema_version": resume_payload.get("e4_checkpoint_schema_version"),
            "atomic_projection_version": ATOMIC_PROJECTION_VERSION,
            "config_sha256": CONFIG_SHA256,
            "model_revision": PINNED_MODEL_REVISION,
            "tokenizer_revision": PINNED_TOKENIZER_REVISION,
            "pretrained_weight_format": WEIGHT_FORMAT.filename,
            "precision_mode": PRECISION.mode,
            "optimizer_signature": OPTIMIZER_SIGNATURE,
            "accumulation_signature": ACCUMULATION.signature,
        })
        base_model.load_state_dict(resume_payload["model_state"]["base_model"])
        head.load_state_dict(resume_payload["model_state"]["w2ner_head"])
        optimizer.load_state_dict(resume_payload["optimizer_state"])
        if scaler is not None and resume_payload.get("scaler_state"):
            scaler.load_state_dict(resume_payload["scaler_state"])
        start_epoch = int(resume_payload["epoch"]) + 1
        optimizer_steps_total = int(resume_payload["optimizer_steps"])
        backward_passes_total = int(resume_payload.get("backward_passes", 0))
        examples_total = int(resume_payload.get("examples_processed", 0))
        best_metric = float(resume_payload["best_metric"])
        best_checkpoint_sha256 = str(resume_payload.get("best_checkpoint_sha256", ""))
    else:
        history_path.write_text("", encoding="utf-8")
    autocast_dtype = torch.bfloat16 if PRECISION.mode == "bf16" else torch.float16
    run_started = time.monotonic()
    rolling_loss = RollingLoss(window=PROGRESS.rolling_loss_window)
    train_samples_total = train_source.example_count
    best_epoch = 0
    latest_checkpoint_sha256 = ""
    current_sample = 0

    for epoch in range(start_epoch, epochs + 1):
      try:
        base_model.train()
        head.train()
        micro_batches = 0
        train_loss = 0.0
        epoch_backward_passes = 0
        epoch_optimizer_steps = 0
        epoch_started = time.monotonic()
        emit(epoch_start_record(
            epoch=epoch, total_epochs=epochs, train_examples=train_samples_total,
            expected_optimizer_steps_this_epoch=ACCUMULATION.optimizer_steps_per_epoch,
            expected_backward_passes_this_epoch=ACCUMULATION.micro_batches_per_epoch,
            resume_start_epoch=start_epoch, precision_mode=PRECISION.mode,
            gpu_name=GPU_NAME), log=PROGRESS_LOG)
        bar = make_progress_bar(train_samples_total, f"train epoch {epoch}/{epochs}",
                                enabled=PROGRESS.progress_bar_enabled)
        optimizer.zero_grad(set_to_none=True)
        # A new independent iterator for every epoch. One contract exists at a
        # time; its O(n^2) grid is released as the loop advances.
        for micro_batch_index, item in enumerate(train_source.iter_contracts()):
            # REAL gradient accumulation: the optimizer steps only at group
            # boundaries, and each micro-batch loss is divided by its group's
            # ACTUAL size so the trailing partial group is not under-scaled.
            group_scale = ACCUMULATION.loss_scale_for(micro_batch_index)
            with torch.autocast(device_type=device.type, dtype=autocast_dtype,
                                enabled=PRECISION.autocast_enabled):
                word_embeddings = _atomic_word_embeddings(base_model, tokenizer, item, device)
                pair_mask = torch.tensor([item.grid.pair_mask], dtype=torch.bool, device=device)
                labels = torch.tensor([item.grid.labels], dtype=torch.long, device=device)
                logits = head(word_embeddings, pair_mask)
                loss = nn.functional.cross_entropy(
                    logits.reshape(-1, item.label_count), labels.reshape(-1))
            scaled_loss = loss * group_scale
            if scaler is not None:
                scaler.scale(scaled_loss).backward()
            else:
                scaled_loss.backward()
            backward_passes_total += 1
            # This float() is the EXISTING mean-loss calculation; the rolling
            # window reuses it, so progress adds no extra CPU/GPU sync.
            sample_loss = float(loss.detach().float().cpu())
            train_loss += sample_loss
            rolling_loss.observe(sample_loss)
            micro_batches += 1
            examples_total += 1
            epoch_backward_passes += 1
            current_sample = micro_batch_index + 1

            if ACCUMULATION.is_optimizer_step_boundary(micro_batch_index):
                if scaler is not None:
                    scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(trainable, MAX_GRAD_NORM)
                if scaler is not None:
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    optimizer.step()
                optimizer.zero_grad(set_to_none=True)
                optimizer_steps_total += 1
                epoch_optimizer_steps += 1
            del item, word_embeddings, pair_mask, labels, logits, loss, scaled_loss

            bar.update(1)
            if should_log_train_sample(PROGRESS, current_sample, train_samples_total):
                epoch_elapsed = time.monotonic() - epoch_started
                bar.set_postfix(**progress_bar_postfix(
                    rolling_mean_loss=rolling_loss.mean,
                    optimizer_steps=epoch_optimizer_steps,
                    samples_per_second=rate_per_second(current_sample, epoch_elapsed),
                    eta_seconds_value=eta_seconds(
                        current_sample, train_samples_total, epoch_elapsed)))
                emit(training_heartbeat(
                    run_mode=mode, epoch=epoch, total_epochs=epochs,
                    sample=current_sample, total_samples=train_samples_total,
                    accumulation_slot=(micro_batch_index % ACCUMULATION.accumulation_steps) + 1,
                    gradient_accumulation_steps=ACCUMULATION.accumulation_steps,
                    epoch_backward_passes=epoch_backward_passes,
                    global_backward_passes=backward_passes_total,
                    epoch_optimizer_steps=epoch_optimizer_steps,
                    global_optimizer_steps=optimizer_steps_total,
                    current_loss=sample_loss, rolling_mean_loss=rolling_loss.mean,
                    epoch_elapsed_seconds=epoch_elapsed,
                    run_elapsed_seconds=time.monotonic() - run_started,
                    learning_rate=float(optimizer.param_groups[0]["lr"]),
                    precision_mode=PRECISION.mode, gpu_name=GPU_NAME,
                    gpu_memory=gpu_memory_snapshot(torch, device.type),
                    memory_snapshot=memory_snapshot), log=PROGRESS_LOG)
            if micro_batch_index == 0 and epoch == start_epoch:
                print(json.dumps({"stage": "memory_after_first_micro_batch",
                                  "memory": memory_snapshot("after_first_micro_batch")},
                                 indent=2, sort_keys=True))
        bar.close()
        epoch_train_elapsed = time.monotonic() - epoch_started
        emit(epoch_training_complete_record(
            epoch=epoch, backward_passes=epoch_backward_passes,
            optimizer_steps=epoch_optimizer_steps,
            mean_training_loss=train_loss / max(1, micro_batches),
            elapsed_seconds=epoch_train_elapsed, samples=micro_batches), log=PROGRESS_LOG)

        validation_metrics = evaluate_w2ner_validation(
            base_model, head, tokenizer, validation_source, device,
            epoch=epoch, total_epochs=epochs)
        best_before_epoch = best_metric
        emit(epoch_validation_complete_record(
            epoch=epoch,
            exact_precision=float(validation_metrics["validation_exact_precision"]),
            exact_recall=float(validation_metrics["validation_exact_recall"]),
            exact_f1=float(validation_metrics["validation_exact_f1"]),
            best_f1_before_epoch=best_before_epoch), log=PROGRESS_LOG)
        row = build_e4_history_row(
            epoch=epoch,
            mode=mode,
            train_loss=train_loss / max(1, micro_batches),
            validation_metrics=validation_metrics,
            optimizer_steps=optimizer_steps_total,
            backward_passes=backward_passes_total,
            examples_processed=examples_total,
            learning_rate=float(optimizer.param_groups[0]["lr"]),
            accumulation=ACCUMULATION,
            precision=PRECISION,
        )
        with history_path.open("a", encoding="utf-8") as handle:
            handle.write(json.dumps(row, sort_keys=True) + "\n")

        payload_kwargs = dict(
            base_model=base_model, head=head, optimizer=optimizer, scaler=scaler,
            epoch=epoch, optimizer_steps=optimizer_steps_total,
            backward_passes=backward_passes_total, examples_processed=examples_total,
            parameter_count=parameter_count, mode=mode,
        )
        persistence_started = time.monotonic()
        best_updated = False
        if float(validation_metrics["validation_exact_f1"]) >= best_metric:
            best_metric = float(validation_metrics["validation_exact_f1"])
            best_epoch = epoch
            best_updated = True
            best_payload = checkpoint_state_payload(
                best_metric=best_metric,
                best_checkpoint_sha256=best_checkpoint_sha256, **payload_kwargs)
            best_checkpoint_sha256 = save_and_persist_checkpoint(
                "best", best_payload, epoch=epoch)
        # latest.pt is rebuilt AFTER best so it records the current best identity;
        # it is a separate payload object, never an alias of best.
        latest_payload = checkpoint_state_payload(
            best_metric=best_metric,
            best_checkpoint_sha256=best_checkpoint_sha256, **payload_kwargs)
        latest_checkpoint_sha256 = save_and_persist_checkpoint(
            "latest", latest_payload, epoch=epoch)
        persist_artifact(
            "training_history.jsonl", history_path,
            OUTPUT_DIR / "logs" / "training_history.jsonl",
            drive_health=check_drive_health)
        # Governed sync point: the local progress log goes to Drive at epoch
        # completion, never per sample.
        persist_artifact(
            "training_progress.jsonl", PROGRESS_LOG_PATH,
            OUTPUT_DIR / "logs" / "training_progress.jsonl",
            drive_health=check_drive_health)
        emit(epoch_checkpoint_persisted_record(
            epoch=epoch,
            latest_checkpoint_path=str(OUTPUT_DIR / "checkpoints" / "latest.pt"),
            latest_checkpoint_sha256=latest_checkpoint_sha256,
            best_checkpoint_updated=best_updated,
            best_checkpoint_path=str(OUTPUT_DIR / "checkpoints" / "best.pt"),
            persistent_custody_verified=True,
            elapsed_seconds=time.monotonic() - persistence_started), log=PROGRESS_LOG)
      except BaseException as training_error:
        # Report, then ALWAYS re-raise. The error is never suppressed.
        emit(training_failed_record(
            epoch=epoch, sample=current_sample,
            global_backward_passes=backward_passes_total,
            global_optimizer_steps=optimizer_steps_total,
            exception=training_error,
            local_progress_log_path=str(PROGRESS_LOG_PATH),
            latest_persistent_checkpoint_available=(
                OUTPUT_DIR / "checkpoints" / "latest.pt").is_file(),
            gpu_memory=gpu_memory_snapshot(torch, device.type),
            memory_snapshot=memory_snapshot), log=PROGRESS_LOG)
        try:
            persist_artifact(
                "training_progress.jsonl", PROGRESS_LOG_PATH,
                OUTPUT_DIR / "logs" / "training_progress.jsonl",
                drive_health=check_drive_health)
        except Exception as sync_error:  # noqa: BLE001 - telemetry is never fatal
            print(json.dumps({"stage": "progress_log_sync_failed",
                              "detail": f"{type(sync_error).__name__}: {sync_error}"},
                             sort_keys=True), flush=True)
        raise
    if best_payload is None or latest_payload is None:
        raise RuntimeError("training finished without checkpoint payloads")
    checkpoint_hashes = {
        name: sha256_file(STAGING_DIR / "checkpoints" / f"{name}.pt")
        for name in ("best", "latest")
    }
    validate_checkpoint_after_save_reload(STAGING_DIR / "checkpoints" / "best.pt", checkpoint_hashes["best"])
    validate_checkpoint_after_save_reload(STAGING_DIR / "checkpoints" / "latest.pt", checkpoint_hashes["latest"])
    validation_metrics = evaluate_w2ner_validation(base_model, head, tokenizer, validation_source, device)
    # The manifest must report what the loop REALLY did, not a relabelled batch
    # size: this raises if the observed counts disagree with the plan.
    training_accounting = build_e4_training_accounting(
        accumulation=ACCUMULATION,
        precision=PRECISION,
        weight_format=WEIGHT_FORMAT,
        observed_optimizer_steps=optimizer_steps_total,
        observed_backward_passes=backward_passes_total,
        observed_examples=examples_total,
        max_grad_norm=MAX_GRAD_NORM,
    )
    assert_optimizer_step_accounting(ACCUMULATION, optimizer_steps_total)
    validation_metrics.update({
        "best_epoch": best_epoch,
        "total_elapsed_seconds": round(time.monotonic() - run_started, 3),
        "progress": {**PROGRESS.as_dict(), **PROGRESS_LOG.as_dict()},
        "checkpoint_hashes": checkpoint_hashes,
        "completed_epochs": epochs,
        "optimizer_steps": optimizer_steps_total,
        "backward_passes": backward_passes_total,
        "examples_processed": examples_total,
        "parameter_count": parameter_count,
        "load_report": load_report,
        "training_accounting": training_accounting,
        "best_metric": best_metric,
        "internal_test_accessed": False,
    })
    return validation_metrics

validation_metrics = run_training(
    TRAIN_SOURCE,
    VALIDATION_SOURCE,
    mode=mode,
    epochs=EPOCHS,
)
METRICS_STAGING = STAGING_DIR / "validation_metrics.json"
METRICS_STAGING.write_text(json.dumps(validation_metrics, indent=2, sort_keys=True) + "\n", encoding="utf-8")
print(json.dumps({
    "stage": "training_completed",
    "mode": mode,
    "validation_exact_f1": validation_metrics["validation_exact_f1"],
    "checkpoint_hashes": validation_metrics["checkpoint_hashes"],
    "optimizer_steps": validation_metrics["optimizer_steps"],
    "backward_passes": validation_metrics["backward_passes"],
    "effective_batch_size": ACCUMULATION.effective_batch_size,
    "precision_mode": PRECISION.mode,
    "pretrained_weight_format": WEIGHT_FORMAT.filename,
    "model_revision": PINNED_MODEL_REVISION,
    "tokenizer_revision": PINNED_TOKENIZER_REVISION,
    "tokenizer_class": type(tokenizer).__name__,
    "tokenizer_is_fast": bool(getattr(tokenizer, "is_fast", False)),
    "internal_test_accessed": False,
}, indent=2, sort_keys=True))

In [ ]:
checkpoint_hashes = dict(validation_metrics["checkpoint_hashes"])
manifest = build_e4_manifest(
    mode=mode,
    status=STATUS_FULLY_TRAINED if RUN_FULL_TRAINING else STATUS_SMOKE_EXECUTED,
    run_completed=True,
    repository_commit=RESOLVED_COMMIT,
    corpus_hashes=corpus_hashes,
    data_hashes=corpus_hashes,
    resolved_config=resolved_config,
    model_revision=PINNED_MODEL_REVISION,
    tokenizer_revision=PINNED_TOKENIZER_REVISION,
    seed=SEED,
    completed_epochs=int(validation_metrics["completed_epochs"]),
    optimizer_steps=int(validation_metrics["optimizer_steps"]),
    effective_batch_size=ACCUMULATION.effective_batch_size,
    parameter_count=int(validation_metrics["parameter_count"]),
    checkpoint_hashes=checkpoint_hashes,
    best_metric=float(validation_metrics["validation_exact_f1"]),
    train_split_id="governed_train_sha256_" + corpus_hashes["train"],
    validation_split_id="governed_validation_sha256_" + corpus_hashes["validation"],
    safe_to_resume=True,
    initialization_source=INITIALIZATION_SOURCE,
    training_accounting={
        **validation_metrics["training_accounting"],
        **MATERIALIZATION,
        **DRIVE_HEALTH.as_dict(),
        "contracts_materialized": False,
        "contract_stream_version": TRAIN_SOURCE.stream_version,
    },
)
manifest.validate()

# STEP 16: every artifact is written locally, verified, then synced to Drive with
# a verified digest. The Hugging Face cache is never part of the artifact.
MANIFEST_STAGING = STAGING_DIR / "training_manifest.json"
manifest.write(MANIFEST_STAGING)
PERSISTED_ARTIFACTS = {}
for _name, _staged, _target in (
    ("resolved_config.json", CONFIG_STAGING, OUTPUT_DIR / "resolved_config.json"),
    ("validation_metrics.json", METRICS_STAGING, OUTPUT_DIR / "validation_metrics.json"),
    ("e4_alignment_diagnostic.json", DIAGNOSTIC_STAGING, OUTPUT_DIR / "e4_alignment_diagnostic.json"),
    ("grid_target_statistics.json", STATISTICS_STAGING, OUTPUT_DIR / "grid_target_statistics.json"),
    ("training_history.jsonl", STAGING_DIR / "logs" / "training_history.jsonl",
     OUTPUT_DIR / "logs" / "training_history.jsonl"),
    ("training_manifest.json", MANIFEST_STAGING, OUTPUT_DIR / "training_manifest.json"),
):
    PERSISTED_ARTIFACTS[_name] = persist_artifact(
        _name, _staged, _target, drive_health=check_drive_health).as_dict()

# STEP 17: the read-only artifact validator runs against the persisted artifact.
report = validate_e4_artifact(OUTPUT_DIR, mode=mode)
print(json.dumps({
    "stage": "artifact_validation",
    "artifact_dir": str(OUTPUT_DIR),
    "manifest_sha256": report.manifest_sha256,
    "validator": report.as_dict(),
    "persisted_artifacts": PERSISTED_ARTIFACTS,
    "drive_health": DRIVE_HEALTH.as_dict(),
    "memory": memory_snapshot("after_artifact_persistence"),
    "status": "SMOKE_EXECUTED" if RUN_SMOKE_TRAINING and not RUN_FULL_TRAINING else "FULLY_TRAINED",
    "internal_test_accessed": False,
}, indent=2, sort_keys=True))
if not report.ok:
    raise AssertionError(report.failures)

# Governed sync point: the local progress log is persisted at clean completion.
persist_artifact(
    "training_progress.jsonl", PROGRESS_LOG_PATH,
    OUTPUT_DIR / "logs" / "training_progress.jsonl",
    drive_health=check_drive_health)
emit(full_training_complete_record(
    epochs_completed=int(validation_metrics["completed_epochs"]),
    global_backward_passes=int(validation_metrics["backward_passes"]),
    global_optimizer_steps=int(validation_metrics["optimizer_steps"]),
    best_validation_exact_f1=float(validation_metrics["best_metric"]),
    best_epoch=int(validation_metrics["best_epoch"]),
    total_elapsed_seconds=float(validation_metrics["total_elapsed_seconds"]),
    total_samples_processed=int(validation_metrics["examples_processed"]),
    best_checkpoint_sha256=checkpoint_hashes["best"],
    latest_checkpoint_sha256=checkpoint_hashes["latest"],
    artifact_validator_ok=bool(report.ok)), log=PROGRESS_LOG)